# PCA Problem Statement Solution

This notebook solves the PCA + clustering problem from `8 PCA_Problem Statement (1).pdf` using `heart disease.xlsx`.

## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

## Load Data and Prepare Features

In [ ]:
df = pd.read_excel('heart disease.xlsx')
print('Shape:', df.shape)
print('Missing:', int(df.isna().sum().sum()))
print('Target distribution:', df['target'].value_counts().to_dict())
X = df.drop(columns=['target'])
X_scaled = StandardScaler().fit_transform(X)

## Select Best K Using Silhouette

In [ ]:
ks = list(range(2, 11))
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(float(km.inertia_))
    sil.append(float(silhouette_score(X_scaled, labels)))
best_k = ks[int(np.argmax(sil))]
print('Best K:', best_k)
pd.DataFrame({'k': ks, 'inertia': inertia, 'silhouette': sil})

## Clustering on Original Data

In [ ]:
km_o = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_km_o = km_o.fit_predict(X_scaled)
hc_o = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
labels_hc_o = hc_o.fit_predict(X_scaled)
print('Original KMeans silhouette:', silhouette_score(X_scaled, labels_km_o))
print('Original Hierarchical silhouette:', silhouette_score(X_scaled, labels_hc_o))

## PCA (3 Components)

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Cumulative (3 PCs):', pca.explained_variance_ratio_.sum())
pca_df = pd.DataFrame(X_pca, columns=['PC1','PC2','PC3'])
pca_df.to_csv('heart_pca_3_components.csv', index=False)
pca_df.head()

## Clustering on PCA Data and Comparison

In [ ]:
km_p = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_km_p = km_p.fit_predict(X_pca)
hc_p = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
labels_hc_p = hc_p.fit_predict(X_pca)
print('PCA KMeans silhouette:', silhouette_score(X_pca, labels_km_p))
print('PCA Hierarchical silhouette:', silhouette_score(X_pca, labels_hc_p))
print('ARI KMeans original vs PCA:', adjusted_rand_score(labels_km_o, labels_km_p))
print('ARI Hierarchical original vs PCA:', adjusted_rand_score(labels_hc_o, labels_hc_p))